In [38]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, multilabel_confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import BallTree
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [22]:
training_df = pd.read_csv(
    filepath_or_buffer='training_faults_diagnostics.csv',
    low_memory=False
)

In [23]:
# Convert SPN and FMI values to strings
training_df['spn'] = training_df['spn'].astype(str)
training_df['fmi'] = training_df['fmi'].astype(str)
print(f'spn datatype: {training_df['spn'].dtype}')

spn datatype: object


In [25]:
target = 'Derate_Target'

# Create dataset with features
X = training_df

# Create array of targets
y = training_df[target]

## Identify features for imputing missing values

In [27]:
# Group categorical columns
categorical_columns = ['EquipmentID', 'spn', 'fmi', 'active', 'Severity_Level']

# Group numeric columns
numeric_columns = [
    'BarometricPressure',
    'EngineCoolantTemperature',
    'EngineLoad',
    'EngineOilPressure',
    'EngineOilTemperature',
    'EngineRpm',
    'FuelRate',
    'FuelTemperature',
    'IntakeManifoldTemperature',
    'Speed',
    'Throttle',
    'TurboBoostPressure'
  ]

## Split training dataset

In [63]:
total_rows = X.shape[0]
training_rows = int(total_rows * 0.6)
validation_rows = int(total_rows * 0.2)
testing_rows = total_rows - training_rows - validation_rows

print(f'60% Training: {training_rows}')
print(f'10% Validation: {validation_rows}')
print(f'20% Testing: {testing_rows}')

60% Training: 634841
10% Validation: 211613
20% Testing: 211615


In [64]:
num_training_rows = training_rows
num_validation_rows = validation_rows

X_train = X.iloc[:num_training_rows]
y_train = y.iloc[:num_training_rows]

X_val = X.iloc[num_training_rows : num_training_rows + num_validation_rows]
y_val = y.iloc[num_training_rows : num_training_rows + num_validation_rows]

X_test = X.iloc[num_training_rows + num_validation_rows :]
y_test = y.iloc[num_training_rows + num_validation_rows :]

# Verify split has the same number of rows as before the split
print(X_train.shape[0] + X_val.shape[0] + X_test.shape[0])
print(training_df.shape[0])

1058069
1058069


## Create pipeline and fit model

In [65]:
categorical_pipe = Pipeline(
    steps=[
        ('categorical_imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore'))
    ]
)

# low_nan_numeric_pipe = Pipeline(
#     steps=[
#         ('scaler', StandardScaler()),
#         ('low_nan_numeric_imputer', SimpleImputer(strategy='median'))
#     ]
# )

numeric_pipe = Pipeline(
    steps=[
        ('scaler', StandardScaler()),
        ('medium_nan_numeric_imputer', IterativeImputer(max_iter=20, random_state=30))
    ]
)

In [66]:
ct = ColumnTransformer(
    transformers=[
        ('categorical_pipe', categorical_pipe, categorical_columns),
        # ('low_nan_numeric_pipe', low_nan_numeric_pipe, low_nan_numeric_columns),
        ('medium_nan_numeric_pipe', numeric_pipe, numeric_columns)
    ]
)

In [67]:
pipe = Pipeline(
    steps=[
        ('transformer', ct),
        ('model', MLPClassifier(
            activation='relu',
            hidden_layer_sizes=(32,32,32)
        ))
    ]
)

In [68]:
pipe.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


Pipeline(steps=[('transformer',
                 ColumnTransformer(transformers=[('categorical_pipe',
                                                  Pipeline(steps=[('categorical_imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['EquipmentID', 'spn', 'fmi',
                                                   'active',
                                                   'Severity_Level']),
                                                 ('medium_nan_numeric_pipe',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler()),
                                                                  ('medium_nan_numeric_imputer',
                                                                   IterativeImputer(max_iter=20,
                                                                                    random_state=30))]),
                                                  ['BarometricPressure',
                                                   'EngineCoolantTemperature',
                                                   'EngineLoad',
                                                   'EngineOilPressure',
                                                   'EngineOilTemperature',
                                                   'EngineRpm', 'FuelRate',
                                                   'FuelTemperature',
                                                   'IntakeManifoldTemperature',
                                                   'Speed', 'Throttle',
                                                   'TurboBoostPressure'])])),
                ('model', MLPClassifier(hidden_layer_sizes=(32, 32, 32)))])

## Adjust Threshold

In [69]:
y_pred_prob_val = pipe.predict_proba(X_val)[:,1]

In [70]:
labels=[0,1,2]

candidate_thresholds = np.arange(
    start=0.01,
    stop=0.925,
    step=0.01
)

thresholds_df = pd.DataFrame({'threshold': candidate_thresholds})
thresholds_df['f1'] = thresholds_df['threshold'].apply(lambda x: f1_score(y_true=y_val, y_pred=y_pred_prob_val >= x, labels=labels, average='macro'))
thresholds_df.sort_values(by='f1', ascending=False, inplace=True)
thresholds_df.head()

,threshold,f1
45,0.46,0.339893
44,0.45,0.339779
43,0.44,0.339723
42,0.43,0.339650
41,0.42,0.339614


## Compare training and testing

In [71]:
threshold = thresholds_df['threshold'].iloc[0]

y_pred_proba_train = pipe.predict_proba(X_train)[:,1]
y_pred_proba_test = pipe.predict_proba(X_test)[:,1]

y_pred_train = y_pred_proba_train >= threshold
y_pred_test = y_pred_proba_test >= threshold

In [72]:
training_cr = classification_report(
    y_true=y_train,
    y_pred=y_pred_train,
    digits=6
)
print(str(training_cr))

training_cm = confusion_matrix(
    y_true=y_train,
    y_pred=y_pred_train,
    labels=labels
)
print(training_cm)

training_mcm = multilabel_confusion_matrix(
    y_true=y_train,
    y_pred=y_pred_train,
    labels=labels
)
print(training_mcm)

              precision    recall  f1-score   support

           0   0.998700  0.999951  0.999325    633872
           1   0.782857  0.310658  0.444805       441
           2   0.000000  0.000000  0.000000       528

    accuracy                       0.998641    634841
   macro avg   0.593852  0.436870  0.481377    634841
weighted avg   0.997720  0.998641  0.998109    634841

[[633841     31      0]
 [   304    137      0]
 [   521      7      0]]
[[[   144    825]
  [    31 633841]]

 [[634362     38]
  [   304    137]]

 [[634313      0]
  [   528      0]]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [73]:
testing_cr = classification_report(
    y_true=y_test,
    y_pred=y_pred_test,
    digits=6
)
print(str(testing_cr))

testing_cm = confusion_matrix(
    y_true=y_test,
    y_pred=y_pred_test
)
print(testing_cm)

testing_mcm = multilabel_confusion_matrix(
    y_true=y_test,
    y_pred=y_pred_test,
    labels=labels
)
print(testing_mcm)

              precision    recall  f1-score   support

           0   0.997393  0.995290  0.996341    211061
           1   0.003003  0.010274  0.004648       292
           2   0.000000  0.000000  0.000000       262

    accuracy                       0.992699    211615
   macro avg   0.333465  0.335188  0.333663    211615
weighted avg   0.994786  0.992699  0.993739    211615

[[210067    994      0]
 [   289      3      0]
 [   260      2      0]]
[[[     5    549]
  [   994 210067]]

 [[210327    996]
  [   289      3]]

 [[211353      0]
  [   262      0]]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
